<div style="background-color:#1F3864; padding:25px; border-radius:8px;">
<h1 style="color:white; text-align:center; margin:0;">🌍 Global Trade Disruption & Commodity Price Prediction</h1>
<p style="color:#D9E2F3; text-align:center; margin-top:10px; font-size:15px;">
Predicting commodity price movements from real-world supply chain disruption signals -
built entirely on live API data (FRED, GDELT, UN Comtrade, World Bank)
</p>
</div>

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import time

## Notebook Overview

This notebook builds a dataset and model to predict commodity price movements —
starting with WTI crude oil — using real-world supply chain disruption signals
sourced directly from public APIs, rather than a synthetic dataset.

**Data sources:**
- **FRED API** — commodity prices (oil WTI, oil Brent, natural gas, wheat, corn, gold, aluminum, iron ore)
- **GDELT** — geopolitical event intensity and global news coverage volume
- **UN Comtrade** — bilateral trade volume between countries
- **World Bank** — country-level logistics performance and economic context

**Target variable:**
Percentage change in WTI crude oil price over a forward window (e.g. 30 days),
predicted from current disruption signals. Predicting the *change* rather than
the raw price level avoids the model simply learning long-term inflation trends,
and instead focuses on how disruption events actually move prices — the same
approach used in the crash prediction project's 63-day-forward target.

**Major real-world events this data is expected to capture:**
COVID-19 Supply Chain Shock (2020), Russia-Ukraine Conflict (2022),
Red Sea Shipping Crisis (2024), Strait of Hormuz Disruption (2026).

<div style="background-color:#EDCC80; padding:15px; border-radius:8px;">
<h2 style="color:#1F3864; text-align:center; margin:0;">1. Create the Data</h2>
</div>

<div style="background-color:#EDEAE5; padding:10px 15px; border-radius:6px;">
<h3 style="color:#1F3864; text-align:center; margin:0;">1.1 Commodities</h3>
</div>

In [2]:
api_key = "a3b1c076b07a348c2f9a24b0799e53e2"

In [3]:
start_date = "2000-01-01"

In [4]:
commodities = {
    "oil_wti": "DCOILWTICO",
    "oil_brent": "DCOILBRENTEU",
    "natural_gas": "DHHNGSP",
    "wheat": "PWHEAMTUSDM",
    "corn": "PMAIZMTUSDM",
    "gold": "IQ12260",
    "aluminum": "PALUMUSDM",
    "iron_ore": "PIORECRUSDM"}

In [5]:
def get_commodity_prices(commodities:dict,start_date:str)-> None :
    for name,series_id in commodities.items():
        url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&observation_start={start_date}&api_key={api_key}&file_type=json"
        response = requests.get(url)

        try:
            data = response.json()
        except ValueError:
            print(f"FAILED: {name} ({series_id}) — empty or invalid response, status {response.status_code}")
            continue

        if "observations" not in data:
            print(f"FAILED: {name} ({series_id}) — {data.get('error_message', 'unknown error')}")
            continue

        df = pd.DataFrame(data["observations"])
        df = df[["date", "value"]]
        df.to_csv(f"data/{name}.csv", index=False)
        print(f"Saved {len(df)} rows to data/{name}.csv")

        time.sleep(0.5)

In [6]:
get_commodity_prices(commodities,start_date)

Saved 6931 rows to data/oil_wti.csv
Saved 6931 rows to data/oil_brent.csv
Saved 6931 rows to data/natural_gas.csv
Saved 318 rows to data/wheat.csv
Saved 318 rows to data/corn.csv
Saved 318 rows to data/gold.csv
Saved 318 rows to data/aluminum.csv
Saved 318 rows to data/iron_ore.csv


<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">1.2 GDELT Events </h3>
</div>

In [35]:
def get_gdelt_events(queries: list) -> None:
    headers = {"User-Agent": "Mozilla/5.0"}
    for query in queries:
        url = "https://api.gdeltproject.org/api/v2/doc/doc"
        params = {
            "query": f'"{query}"',
            "mode": "timelinevol",
            "format": "json"
        }

        response = requests.get(url, params=params, headers=headers)
        attempts = 0
        while response.status_code == 429 and attempts < 4:
            wait_time = 15 * (attempts + 1)
            print(f"Rate limited on {query}, waiting {wait_time}s (attempt {attempts+1})...")
            time.sleep(wait_time)
            response = requests.get(url, params=params, headers=headers)
            attempts += 1

        try:
            data = response.json()
        except ValueError:
            print(f"FAILED: {query} — status {response.status_code}, invalid response")
            continue

        if "timeline" not in data:
            print(f"FAILED: {query} — keys were {list(data.keys())}")
            continue

        records = data["timeline"][0]["data"]
        df = pd.DataFrame(records)
        filename = query.lower().replace(" ", "_")
        df.to_csv(f"data/gdelt_{filename}.csv", index=False)
        print(f"Saved data/gdelt_{filename}.csv")
        time.sleep(10)
events = ["Strait of Hormuz", "Red Sea shipping", "Russia Ukraine conflict", "COVID supply chain"]

In [36]:
get_gdelt_events(events)

Saved data/gdelt_strait_of_hormuz.csv
Rate limited on Red Sea shipping, waiting 15s (attempt 1)...
Rate limited on Red Sea shipping, waiting 30s (attempt 2)...
Rate limited on Red Sea shipping, waiting 45s (attempt 3)...
Saved data/gdelt_red_sea_shipping.csv
Rate limited on Russia Ukraine conflict, waiting 15s (attempt 1)...
Rate limited on Russia Ukraine conflict, waiting 30s (attempt 2)...
Saved data/gdelt_russia_ukraine_conflict.csv
Rate limited on COVID supply chain, waiting 15s (attempt 1)...
Rate limited on COVID supply chain, waiting 30s (attempt 2)...
Saved data/gdelt_covid_supply_chain.csv


<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">1.3 UN-Comtrade </h3>
</div>

In [10]:
primary_key = "fb672033c45d4d71a329a740d788c758"

In [11]:
reporters = {
    "egypt": "818",
    "iran": "364",
    "china": "156",
    "usa": "842"}

In [12]:
def get_comtrade_data(reporters: dict, start_year: int, end_date: int) -> None:
    url = "https://comtradeapi.un.org/data/v1/get/C/A/HS"
    headers = {"Ocp-Apim-Subscription-Key": primary_key}

    all_years = list(range(start_year, end_date + 1))
    year_chunks = [all_years[i:i+12] for i in range(0, len(all_years), 12)]

    for name, reporter_code in reporters.items():
        all_records = []

        for chunk in year_chunks:
            years_str = ",".join(str(y) for y in chunk)
            params = {
                "reporterCode": reporter_code,
                "partnerCode": "0",
                "period": years_str,
                "cmdCode": "TOTAL",
                "flowCode": "M"
            }

            response = requests.get(url, headers=headers, params=params)

            if response.status_code == 429:
                print(f"Rate limited on {name}, waiting...")
                time.sleep(3)
                response = requests.get(url, headers=headers, params=params)

            try:
                data = response.json()
            except ValueError:
                print(f"FAILED chunk {years_str} for {name} — invalid response")
                continue

            if "data" not in data:
                print(f"FAILED chunk {years_str} for {name} — {data.get('error')}")
                continue

            all_records.extend(data["data"])
            time.sleep(1.5)

        df = pd.DataFrame(all_records)
        df.to_csv(f"data/comtrade_{name}.csv", index=False)
        print(f"Saved {len(df)} rows to data/comtrade_{name}.csv")

In [13]:
get_comtrade_data(reporters, start_year=2000, end_date = 2027)

Saved 26 rows to data/comtrade_egypt.csv
Saved 19 rows to data/comtrade_iran.csv
Saved 658 rows to data/comtrade_china.csv
Saved 26 rows to data/comtrade_usa.csv


<div style="background-color:#EDCC80; padding:15px; border-radius:8px;">
<h2 style="color:#1F3864; text-align:center; margin:0;">2. Load the data </h2>
</div>

<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">2.1 Commodities </h3>
</div>

In [14]:
oil_wti = pd.read_csv("data/oil_wti.csv")
oil_brent = pd.read_csv("data/oil_brent.csv")
natural_gas = pd.read_csv("data/natural_gas.csv")
wheat = pd.read_csv("data/wheat.csv")
corn = pd.read_csv("data/corn.csv")
gold = pd.read_csv("data/gold.csv")
aluminum = pd.read_csv("data/aluminum.csv")
iron_ore = pd.read_csv("data/iron_ore.csv")

<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">2.2 gdelt </h3>
</div>

In [15]:
gdelt_hormuz = pd.read_csv("data/gdelt_strait_of_hormuz.csv")
gdelt_red_sea = pd.read_csv("data/gdelt_red_sea_shipping.csv")
gdelt_russia_ukraine = pd.read_csv("data/gdelt_russia_ukraine_conflict.csv")
gdelt_covid = pd.read_csv("data/gdelt_covid_supply_chain.csv")

<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">2.3 UN-Comtrade </h3>
</div>

In [16]:
comtrade_egypt = pd.read_csv("data/comtrade_egypt.csv")
comtrade_iran = pd.read_csv("data/comtrade_iran.csv")
comtrade_china = pd.read_csv("data/comtrade_china.csv")
comtrade_usa = pd.read_csv("data/comtrade_usa.csv")

<div style="background-color:#EDCC80; padding:15px; border-radius:8px;">
<h2 style="color:#1F3864; text-align:center; margin:0;">3. Merge the data </h2>
</div>

<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">3.1 Commodities </h3>
</div>

In [17]:
commodities = oil_wti.rename(columns={"value": "oil_wti"})

In [18]:
commodities = pd.merge(commodities, oil_brent.rename(columns={"value": "oil_brent"}), on="date", how="left")
commodities = pd.merge(commodities, natural_gas.rename(columns={"value": "natural_gas"}), on="date", how="left")
commodities = pd.merge(commodities, wheat.rename(columns={"value": "wheat"}), on="date", how="left")
commodities = pd.merge(commodities, corn.rename(columns={"value": "corn"}), on="date", how="left")
commodities = pd.merge(commodities, gold.rename(columns={"value": "gold"}), on="date", how="left")
commodities = pd.merge(commodities, aluminum.rename(columns={"value": "aluminum"}), on="date", how="left")
commodities = pd.merge(commodities, iron_ore.rename(columns={"value": "iron_ore"}), on="date", how="left")

<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">3.2 gdelt </h3>
</div>

In [19]:
gdelt = gdelt_hormuz.rename(columns={"value": "hormuz_risk"})

In [20]:
gdelt = pd.merge(gdelt, gdelt_red_sea.rename(columns={"value": "red_sea_risk"}), on="date", how="left")
gdelt = pd.merge(gdelt, gdelt_russia_ukraine.rename(columns={"value": "russia_ukraine_risk"}), on="date", how="left")
gdelt = pd.merge(gdelt, gdelt_covid.rename(columns={"value": "covid_risk"}), on="date", how="left")


<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">3.3 UN-Comtrade </h3>
</div>

In [21]:
comtrade = comtrade_egypt[["period","primaryValue"]].rename(columns={"primaryValue": "egypt_trade_value"})
comtrade = pd.merge(comtrade, comtrade_iran[["period","primaryValue"]].rename(columns={"primaryValue": "iran_trade_value"}))
comtrade = pd.merge(comtrade, comtrade_china[["period","primaryValue"]].rename(columns={"primaryValue": "china_trade_value"}))
comtrade = pd.merge(comtrade, comtrade_usa[["period","primaryValue"]].rename(columns={"primaryValue": "usa_trade_value"}))

<div style="background-color:#EDCC80; padding:15px; border-radius:8px;">
<h2 style="color:#1F3864; text-align:center; margin:0;">4. prepare the data </h2>
</div>

In [22]:
commodities

,date,oil_wti,oil_brent,natural_gas,wheat,corn,gold,aluminum,iron_ore
0,2000-01-03,.,.,.,NaN,NaN,NaN,NaN,NaN
1,2000-01-04,25.56,23.95,2.16,NaN,NaN,NaN,NaN,NaN
2,2000-01-05,24.65,23.72,2.17,NaN,NaN,NaN,NaN,NaN
3,2000-01-06,24.79,23.55,2.18,NaN,NaN,NaN,NaN,NaN
4,2000-01-07,24.79,23.35,2.19,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
6926,2026-07-21,86.04,93.85,2.8,NaN,NaN,NaN,NaN,NaN
6927,2026-07-22,87.66,94.12,2.93,NaN,NaN,NaN,NaN,NaN
6928,2026-07-23,93.08,105.32,2.92,NaN,NaN,NaN,NaN,NaN
6929,2026-07-24,91.74,100.31,2.87,NaN,NaN,NaN,NaN,NaN


In [23]:
gdelt

,date,hormuz_risk,red_sea_risk,russia_ukraine_risk,covid_risk
0,20260510T000000Z,1.5867,0.0027,0.0224,0.0000
1,20260511T000000Z,1.6899,0.0000,0.0026,0.0007
2,20260512T000000Z,1.6532,0.0006,0.0068,0.0006
3,20260513T000000Z,1.3163,0.0006,0.0056,0.0000
4,20260514T000000Z,1.2905,0.0019,0.0089,0.0000
...,...,...,...,...,...
76,20260729T000000Z,0.9075,0.0148,0.0123,0.0000
77,20260730T000000Z,0.9842,0.0119,0.0036,0.0000
78,20260731T000000Z,0.5097,0.0127,0.0160,0.0000
79,20260801T000000Z,0.6022,0.0091,0.0100,0.0000


In [24]:
comtrade

,period,egypt_trade_value,iran_trade_value,china_trade_value,usa_trade_value
0,2000,1.401780e+10,1.362601e+10,2.250937e+11,1.217933e+12
1,2001,1.277947e+10,1.617308e+10,2.435529e+11,1.140900e+12
2,2002,1.255243e+10,2.033575e+10,2.951701e+11,1.200096e+12
3,2003,1.123051e+10,2.563812e+10,4.127598e+11,1.302834e+12
4,2004,1.286434e+10,3.299707e+10,5.612287e+11,1.525304e+12
...,...,...,...,...,...
647,2018,8.190952e+10,4.123617e+10,2.133605e+12,2.611432e+12
648,2019,7.651492e+10,4.397557e+10,2.079285e+12,2.567492e+12
649,2020,7.043679e+10,3.880458e+10,2.069568e+12,2.405382e+12
650,2021,8.920620e+10,5.295797e+10,2.679412e+12,2.932976e+12


## First is we clean 'Date' columns and Merge the date into one dateset

In [25]:
commodities["date"]=pd.to_datetime(commodities["date"])
gdelt["date"]=pd.to_datetime(gdelt["date"]).dt.tz_localize(None)
comtrade["period"]=pd.to_datetime(comtrade["period"],format = "%Y")

In [26]:
comtrade = comtrade.rename(columns={"period":"date"}) 

In [27]:
comtrade

,date,egypt_trade_value,iran_trade_value,china_trade_value,usa_trade_value
0,2000-01-01,1.401780e+10,1.362601e+10,2.250937e+11,1.217933e+12
1,2001-01-01,1.277947e+10,1.617308e+10,2.435529e+11,1.140900e+12
2,2002-01-01,1.255243e+10,2.033575e+10,2.951701e+11,1.200096e+12
3,2003-01-01,1.123051e+10,2.563812e+10,4.127598e+11,1.302834e+12
4,2004-01-01,1.286434e+10,3.299707e+10,5.612287e+11,1.525304e+12
...,...,...,...,...,...
647,2018-01-01,8.190952e+10,4.123617e+10,2.133605e+12,2.611432e+12
648,2019-01-01,7.651492e+10,4.397557e+10,2.079285e+12,2.567492e+12
649,2020-01-01,7.043679e+10,3.880458e+10,2.069568e+12,2.405382e+12
650,2021-01-01,8.920620e+10,5.295797e+10,2.679412e+12,2.932976e+12


## Merge into One dataset

In [28]:
master = pd.merge(commodities,gdelt,on="date",how="left")

In [30]:
master = pd.merge(master,comtrade,on="date",how="left")

In [37]:
master

,date,oil_wti,oil_brent,natural_gas,wheat,corn,gold,aluminum,iron_ore,hormuz_risk,red_sea_risk,russia_ukraine_risk,covid_risk,egypt_trade_value,iran_trade_value,china_trade_value,usa_trade_value
0,2000-01-03,.,.,.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-04,25.56,23.95,2.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-05,24.65,23.72,2.17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-06,24.79,23.55,2.18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-07,24.79,23.35,2.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7343,2026-07-21,86.04,93.85,2.8,NaN,NaN,NaN,NaN,NaN,1.0716,0.0314,0.0130,0.0000,NaN,NaN,NaN,NaN
7344,2026-07-22,87.66,94.12,2.93,NaN,NaN,NaN,NaN,NaN,1.1477,0.0258,0.0067,0.0000,NaN,NaN,NaN,NaN
7345,2026-07-23,93.08,105.32,2.92,NaN,NaN,NaN,NaN,NaN,1.0470,0.0467,0.0068,0.0006,NaN,NaN,NaN,NaN
7346,2026-07-24,91.74,100.31,2.87,NaN,NaN,NaN,NaN,NaN,1.0043,0.0356,0.0082,0.0021,NaN,NaN,NaN,NaN


In [38]:
master.isna().sum()

date                      0
oil_wti                   0
oil_brent                 0
natural_gas               0
wheat                  6705
corn                   6705
gold                   6705
aluminum               6705
iron_ore               6705
hormuz_risk            7295
red_sea_risk           7295
russia_ukraine_risk    7295
covid_risk             7295
egypt_trade_value      6918
iran_trade_value       6918
china_trade_value      6918
usa_trade_value        6918
dtype: int64

In [42]:
master["wheat"] = commodities["wheat"].ffill()
master["corn"] = commodities["corn"].ffill()
master["gold"] = commodities["gold"].ffill()
master["aluminum"] = commodities["aluminum"].ffill()
master["iron_ore"] = commodities["iron_ore"].ffill()

In [43]:
master.isna().sum()

date                      0
oil_wti                   0
oil_brent                 0
natural_gas               0
wheat                   438
corn                    438
gold                    438
aluminum                438
iron_ore                438
hormuz_risk            7295
red_sea_risk           7295
russia_ukraine_risk    7295
covid_risk             7295
egypt_trade_value      6918
iran_trade_value       6918
china_trade_value      6918
usa_trade_value        6918
dtype: int64

In [44]:
master

,date,oil_wti,oil_brent,natural_gas,wheat,corn,gold,aluminum,iron_ore,hormuz_risk,red_sea_risk,russia_ukraine_risk,covid_risk,egypt_trade_value,iran_trade_value,china_trade_value,usa_trade_value
0,2000-01-03,.,.,.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-04,25.56,23.95,2.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-05,24.65,23.72,2.17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-06,24.79,23.55,2.18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-07,24.79,23.35,2.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7343,2026-07-21,86.04,93.85,2.8,NaN,NaN,NaN,NaN,NaN,1.0716,0.0314,0.0130,0.0000,NaN,NaN,NaN,NaN
7344,2026-07-22,87.66,94.12,2.93,NaN,NaN,NaN,NaN,NaN,1.1477,0.0258,0.0067,0.0000,NaN,NaN,NaN,NaN
7345,2026-07-23,93.08,105.32,2.92,NaN,NaN,NaN,NaN,NaN,1.0470,0.0467,0.0068,0.0006,NaN,NaN,NaN,NaN
7346,2026-07-24,91.74,100.31,2.87,NaN,NaN,NaN,NaN,NaN,1.0043,0.0356,0.0082,0.0021,NaN,NaN,NaN,NaN


## Merge Summary

Combined three real-world data sources into a single master dataset, aligned by date:

- **Commodities (FRED)** — daily oil (WTI, Brent), natural gas; monthly wheat, corn, gold, aluminum, iron ore. Monthly series forward-filled so every day carries the most recent known value.
- **GDELT Events** — daily geopolitical risk scores for four disruption topics (Strait of Hormuz, Red Sea shipping, Russia-Ukraine conflict, COVID supply chain).
- **UN Comtrade** — yearly bilateral trade volume for Egypt, Iran, China, and USA, merged on `year` rather than `date` (Comtrade only reports annually).

**Known, expected gaps in the data — not bugs:**
- **Commodities**: a small number of NaN rows at the very start of the date range, before each monthly-reported commodity's first recorded value.
- **GDELT**: heavily NaN outside roughly the last 3 months, since GDELT's DOC API only provides a limited historical lookback window. This means GDELT's signal is only present for the most recent slice of the dataset — including our the most recent event (Strait of Hormuz, 2026) but NOT the older ones (COVID 2020, Russia-Ukraine 2022), which fall outside its coverage.
- **UN Comtrade**: NaN before 2000 and in years without complete reporting for a given country.

**Decision point for modeling:** GDELT's limited historical coverage means it can only meaningfully inform predictions for the recent period. This will be handled either by filling remaining NaNs with 0 (treating missing data as "no elevated signal") or leaving them for models that natively handle missing values (CatBoost, LightGBM).

<div style="background-color:#EDCC80; padding:15px; border-radius:8px;">
<h2 style="color:#1F3864; text-align:center; margin:0;">5. Explanatory Data Analysis(EDA) </h2>
</div>